In [32]:
from dotenv import load_dotenv
import os

from langchain.memory import ConversationBufferMemory, ConversationBufferWindowMemory

load_dotenv()
print(f"API_KEY: {os.getenv('API_KEY')}")
print(f"API_URL: {os.getenv('API_URL')}")
print(f"API_MODEL: {os.getenv('API_MODEL')}")

API_KEY: sk-d04de33360574dd6bd8224dc1d9897a3
API_URL: https://dashscope.aliyuncs.com/compatible-mode/v1
API_MODEL: qwen3-32b


In [49]:
from dataclasses import Field

from langchain_core.output_parsers import PydanticOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, trim_messages
from langchain_core.prompts import ChatPromptTemplate

import os

from pydantic import BaseModel, Field

api_key = os.getenv('API_KEY')
api_url = os.getenv('API_URL')
default_model = os.getenv('API_MODEL')
default_temperature = os.getenv('API_TEMPERATURE',0)

# llm输出格式
class FinAnswer(BaseModel):
    answer: str = Field(description="答案选项")
    reason: str = Field(description="选择原因")

parser = PydanticOutputParser(pydantic_object=FinAnswer)

def build_llm(llm_model=default_model, **args):
    chat_model = ChatOpenAI(
        model=default_model,
        api_key=api_key,
        base_url=api_url,
        temperature=args['temperature'] if 'temperature' in args else default_temperature,
        **{k: v for k, v in args.items() if k!='temperature'})
    return chat_model
args = {"extra_body": {"enable_thinking": False}}
llm = build_llm('qwen-turbo', **args)

def call_llm(question:str):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "你是一个{role}。你的任务是根据用户的问题给出明确的答案。\n{response_instructions}"),
        ("human", "{question}")
    ])
    chain = prompt | llm | parser
    response = chain.invoke({
        "role": "专业的基金投研分析师",
        "response_instructions": parser.get_format_instructions(),
        "question": question})
    return response


In [51]:
import json

import pandas as pd
fin_df = pd.read_csv('findata/val/finance_val.csv')

results = []
for row in fin_df.itertuples():
    test = {
        "question": row.question,
        "options": {
            "A": row.A,
            "B": row.B,
            "C": row.C,
            "D": row.D
        }
    }
    response:FinAnswer = call_llm(json.dumps(test, ensure_ascii=False))
    print(response)
    test['llm_answer'] = response.answer
    test['expect_answer'] = row.answer
    test['result'] = 1 if test['llm_answer'] == test['expect_answer'] else 0
    results.append(test)

df = pd.DataFrame(results)
df


answer='B' reason='直接信用是指资金需求者通过金融市场直接向资金供给者融资，如发行股票或债券。而选项B中，信用合作社对农户发放贷款属于间接信用，因为资金的融通是通过金融机构（信用合作社）作为中介完成的，而非直接在市场中进行。'
answer='C' reason='根据费雪方程式，名义利率近似等于实际利率加上预期通货膨胀率。因此，3% + 6% = 9%，所以答案是选项 C。'
answer='D' reason='在我国，"厘"是利率单位，1厘等于0.1%。因此年息3厘即年利率为3%，月息2厘即月利率为0.2%，拆息2厘通常指日利率为0.02%。'
answer='A' reason='挂牌公告的定期储蓄和活期储蓄存款利率是由银行等金融机构根据央行指导设定的，属于官方规定的利率（官定利率），而不是市场自由形成的市场利率。'
answer='B' reason='永久性债券的理论市场价格可以通过公式计算：市场价格 = 年利息 / 市场利率。年利息为100元 × 10% = 10元，市场利率为8%，所以市场价格 = 10元 / 8% = 125元。因此，答案是选项B。'
answer='A' reason='银行为该企业开具银行汇票，是在为企业提供信用支持，体现了银行作为信用中介的职能。'
answer='A' reason='期货是一种标准化的合约，其价值直接与标的资产的价格挂钩，没有内在的期权特征（如行权价、行权时间等）。而信用期权、可召回债券和认股权证都具有类似期权的特性，例如选择权或条件性执行。'
answer='D' reason='期货交易的真正目的是多方面的，包括减少交易者所承担的风险（套期保值）、转让实物资产或金融资产的财产权（交割），以及作为一种商品交换的工具（投机和套利）。因此，选项D是正确的。'
answer='C' reason='金融衍生工具的基本功能包括转移价格风险（A）、形成权威性价格（B）和提高资产管理质量（D）。而保持资信度不变（C）并不是金融衍生工具的功能，它更多与企业的信用管理和财务政策相关。'
answer='B' reason='世界贸易组织(WTO)是一个处理全球贸易规则的国际组织，而不是一个金融机构。而国际货币基金组织(IMF)、国际复兴开发银行(IBRD)和国际清算银行(BIS)都是典型的国际性金融机构

,question,options,llm_answer,expect_answer,result
0,下列信用形式中，不属于直接信用的有____。,"{'A': '商业企业对客户赊销商品', 'B': '信用合作社对农户发放贷款', 'C':...",B,B,1
1,实际利率为3$\%$，预期通货膨胀率为6$\%$，则名义利率水平应该近似的等于____。,"{'A': '2$\%$', 'B': '3$\%$', 'C': '9$\%$', 'D'...",C,C,1
2,我国习惯上将年息、月息、拆息都以“厘”做单位，若年息3厘、月息2厘、拆息2厘，则分别是指____。,"{'A': '年利率为3$\%$，月利率为0.02$\%$，日利率为0.2$\%$', 'B...",D,D,1
3,挂牌公告的定期储蓄和活期储蓄存款利率不属于____。,"{'A': '市场利率', 'B': '名义利率', 'C': '固定利率', 'D': '...",A,A,1
4,面值为100元的永久性债券票面利率是10$\%$，当市场利率为8$\%$时，该债券的理论市场...,"{'A': '100', 'B': '125', 'C': '110', 'D': '1375'}",B,B,1
5,某企业因业务需要，申请银行为其开具银行汇票，银行经审查后，同意企业的申请，为其开具了一张10...,"{'A': '信用中介职能', 'B': '支付中介职能', 'C': '化货币收入为资本职...",A,D,0
6,以下的金融资产中不具有与期权类似的特征的是____。,"{'A': '期货', 'B': '信用期权可召回债券认股权证', 'C': '可召回债券'...",A,A,1
7,期货交易的真正目的是____。,"{'A': '作为一种商品交换的工具', 'B': '减少交易者所承担的风险', 'C': ...",D,B,0
8,以下不属于金融衍生工具的基本功能的____。,"{'A': '转移价格风险', 'B': '形成权威性价格', 'C': '保持资信度不变'...",C,C,1
9,以下哪个组织不属于国际性金融机构____。,"{'A': '国际货币基金组织(IMF)', 'B': '世界贸易组织(WTO)', 'C'...",B,B,1
